# Buckets and warmup
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week6/Buckets_And_Warmup_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 6, Module 5: The serving runtime**  
**Author:** Dr. Kaoutar El Maghraoui

A model compiled for fixed shapes cannot run a request of any length. The serving runtime pads each request up to one of a few compiled lengths, called **buckets**. Coarse buckets mean fewer graphs and more padding. Fine buckets mean less padding and more graphs to compile and keep. **Warmup** compiles every bucket before the first request arrives, so no user waits for a compile. A **compile guard** checks that nothing compiles after warmup, and fails loudly if a request needs a shape that was never compiled.

In this notebook you measure the padding cost of two bucket sets on a stream of requests, serve the stream with and without warmup, and add a compile guard that catches a missing bucket.

**What you will do**
1. Reproduce the lecture's padding example: one 600-token request on two bucket sets.
2. Measure the padding cost of both bucket sets on 40 requests.
3. Serve the requests without warmup and see the latency spikes.
4. Warm up every bucket first and serve again.
5. Add a compile guard and trigger it with a missing bucket.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. The notebook takes about three minutes, most of it compiling buckets.

## 1. Padding for one request
The lecture's example is one 600-token request. With a coarse set of two buckets it pads to 2048. With a finer set of five it pads to 1024.

In [ ]:
import math
import random
import statistics
import time
import torch
import torch.nn as nn

COARSE = [512, 2048]
FINE = [128, 256, 512, 1024, 2048]


def bucket_for(n, buckets):
    for b in buckets:
        if n <= b:
            return b
    raise ValueError(f"no bucket holds a {n}-token request (largest is {buckets[-1]})")


for name, buckets in (("coarse", COARSE), ("fine", FINE)):
    b = bucket_for(600, buckets)
    print(f"{name:6} {buckets}: a 600-token request pads to {b}, {(b - 600) / b:.0%} of the work is padding")

## 2. Padding on a stream of requests
Real requests are not all 600 tokens. The cell draws 40 request lengths from a long-tailed distribution with a median of about 300 tokens and measures the padding with each bucket set.

In [ ]:
rng = random.Random(0)
lengths = [min(int(rng.lognormvariate(math.log(300), 0.8)) + 8, 2000) for _ in range(40)]
print(f"40 requests, median {statistics.median(lengths):.0f} tokens, longest {max(lengths)}")

for name, buckets in (("coarse", COARSE), ("fine", FINE)):
    padded = sum(bucket_for(n, buckets) for n in lengths)
    used = {bucket_for(n, buckets) for n in lengths}
    print(f"{name:6}: {1 - sum(lengths) / padded:.0%} of the tokens processed are padding; "
          f"buckets used {sorted(used)}, graphs to compile {len(buckets)}")

**Reading the output**
- With two buckets, about half of the work is padding, because most requests are short and pad to 512 or 2048.
- With five buckets, the padding drops a lot, at the cost of more graphs to compile and hold.
- The lecture's advice: choose the buckets from the measured length distribution of real requests.

## 3. A small model compiled for fixed shapes
Two transformer layers with causal attention, model width 256. A prefill call takes a padded batch of one request and returns the next-token logits for the last real position. With causal attention, the padding at the end of the request does not change the logits at the real positions. The function is compiled with `dynamic=False`, so each bucket is a separate graph.

The compile cache is turned off so that every compile in this notebook is a real compile. Section 6 comes back to the cache.

In [ ]:
assert torch.cuda.is_available(), "This notebook needs a GPU: Runtime > Change runtime type > T4 GPU"
import torch._inductor.config as inductor_config
inductor_config.force_disable_caches = True

import torch.nn.functional as F

D, HEADS = 256, 8


class Layer(nn.Module):
    def __init__(self):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(D), nn.LayerNorm(D)
        self.qkv, self.o = nn.Linear(D, 3 * D), nn.Linear(D, D)
        self.up, self.down = nn.Linear(D, 4 * D), nn.Linear(4 * D, D)

    def forward(self, x):
        B, T, _ = x.shape
        q, k, v = self.qkv(self.n1(x)).view(B, T, 3, HEADS, D // HEADS).permute(2, 0, 3, 1, 4)
        att = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.o(att.transpose(1, 2).reshape(B, T, D))
        return x + self.down(F.gelu(self.up(self.n2(x))))


torch.manual_seed(0)
emb = nn.Embedding(8000, D).cuda().eval()
layers = nn.ModuleList(Layer() for _ in range(2)).cuda().eval()
head = nn.Linear(D, 8000).cuda().eval()


def prefill(tokens, last):
    h = emb(tokens)
    for layer in layers:
        h = layer(h)
    return head(h[0, last])


compiled_prefill = torch.compile(prefill, dynamic=False)


def serve(n, buckets):
    """Pad one request of n tokens to its bucket and run the compiled prefill."""
    b = bucket_for(n, buckets)
    tokens = torch.zeros(1, b, dtype=torch.long, device="cuda")
    tokens[0, :n] = torch.randint(1, 8000, (n,), device="cuda")
    last = torch.tensor(n - 1, device="cuda")
    with torch.inference_mode():
        return compiled_prefill(tokens, last)


def timed(fn):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1e3

## 4. Serving without warmup
The cell serves the 40 requests with the fine buckets, starting from an empty compile cache. The first request for each bucket compiles that bucket while the user waits.

In [ ]:
torch._dynamo.reset()
seen, no_warmup = set(), []
for n in lengths:
    b = bucket_for(n, FINE)
    ms = timed(lambda: serve(n, FINE))
    no_warmup.append(ms)
    flag = "  <- compiled this bucket" if b not in seen else ""
    seen.add(b)
    if flag or len(no_warmup) <= 3:
        print(f"request {len(no_warmup):2d}: {n:4d} tokens -> bucket {b:4d}, {ms:8.1f} ms{flag}")
print(f"\nmedian latency {statistics.median(no_warmup):.2f} ms, worst {max(no_warmup):,.0f} ms")

**Reading the output**
- Each request that is the first in its bucket takes seconds: that user waited for a compile.
- The other requests take a few milliseconds.
- The median looks fine and the worst case is terrible. A server measured only by its median would hide this.

## 5. Warmup, then serve
The runtime lists every bucket, compiles each one before serving, and then serves. The cell times the warmup separately.

In [ ]:
torch._dynamo.reset()
t0 = time.perf_counter()
for b in FINE:
    serve(b, FINE)                       # one request of exactly the bucket length compiles it
torch.cuda.synchronize()
warmup_s = time.perf_counter() - t0
print(f"warmup: {len(FINE)} buckets compiled in {warmup_s:.1f} s")

with_warmup = [timed(lambda: serve(n, FINE)) for n in lengths]
print(f"median latency {statistics.median(with_warmup):.2f} ms, worst {max(with_warmup):.2f} ms")

**Reading the output**
- Warmup pays all the compiles once, before any user arrives.
- After warmup, the worst latency is within a few milliseconds of the median. No request compiles.
- In production the compiled graphs are kept in a persisted cache, so a restart does not recompile them. This notebook turned that cache off to show the real compile cost.

## 6. A compile guard
A request that needs a shape that was never compiled still runs: it compiles on the spot, and that user waits. The lecture's compile guard checks the compiler's counter around each serving call and raises an error if anything compiled after warmup. In continuous integration it is set to fail the build, so a change that leaves a shape uncovered is caught before it ships.

The cell warms up only the three smallest buckets, then serves requests of 200 and 900 tokens.

In [ ]:
from torch._dynamo.utils import counters


class CompileAfterWarmup(RuntimeError):
    pass


def compiles_so_far():
    return counters["stats"]["unique_graphs"]


def guarded_serve(n, buckets, baseline):
    out = serve(n, buckets)
    if compiles_so_far() != baseline:
        raise CompileAfterWarmup(f"a {n}-token request compiled bucket {bucket_for(n, buckets)} after warmup")
    return out


torch._dynamo.reset()
counters.clear()
for b in FINE[:3]:
    serve(b, FINE)                       # warm up 128, 256 and 512 only
baseline = compiles_so_far()
print(f"warmed up {FINE[:3]}: {baseline} compiled graphs")

for n in (200, 900):
    try:
        guarded_serve(n, FINE, baseline)
        print(f"{n}-token request: served from a warmed bucket")
    except CompileAfterWarmup as err:
        print(f"{n}-token request: {err}")

**Reading the output**
- The 200-token request uses the warmed 256 bucket, and the guard is quiet.
- The 900-token request needs the 1024 bucket, which was never warmed up. The guard raises an error that names the request and the bucket, instead of letting a slow request pass unnoticed.
- This is the performance side of the missed-bucket bug from Week 5.

## Try this
1. In section 2, add a 768 bucket to the fine set. How much padding does it save, and what does it cost at warmup?
2. In section 1, find the request length with the worst padding for each bucket set.
3. Turn the compile cache back on (`force_disable_caches = False`), run section 5 twice, and compare the warmup times. This is the persisted graph cache from the lecture.

## Summary
- Fixed-shape compilation pads each request to a bucket. Coarse buckets waste work, fine buckets cost compiles.
- Without warmup, the first request in each bucket waits for a compile. The median hides it, and the worst case does not.
- Warmup compiles every bucket before serving. A persisted cache keeps the graphs across restarts.
- A compile guard turns a missing bucket into an error instead of a slow request.

Lecture: Week 6, Module 5, slides 62 to 64 (bucketing trades waste for warmup, compile at warmup, prefill and decode need different buckets).